In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
import seaborn as sns
from collections import Counter

In [2]:
file_0529 = "../output/0529.csv"
file_0610 = "../output/0610.csv"
df_0529 = pd.read_csv(file_0529)
df_0610 = pd.read_csv(file_0610)

In [3]:
json_0529 = "../output/bar_configs_0529.json"
json_0610 = "../output/bar_configs_0610.json"

In [4]:
with open(json_0529, 'r') as file:
    data_0529 = json.load(file)
    data_0529_dict = {f"{item["well"]}_fld-{item['field']}_{item["id"]}": item for item in data_0529}

In [5]:
with open(json_0610, 'r') as file:
    data_0610 = json.load(file)
    data_0610_dict = {f"{item["well"]}_fld-{item['field']}_{item["id"]}": item for item in data_0610}

In [6]:
field_set_0610 = df_0610["field_name"].unique()
field_set_0529 = df_0529["field_name"].unique()
common_fields = np.intersect1d(list(field_set_0529), list(field_set_0610))

In [ ]:
field_configs_0529 = []
field_configs_0610 = []

field_correspondence = {}

for common_field in common_fields:
    well = common_field.split("_")[0]
    field = int(common_field.split("_")[1][4:])
    label_map_0529 = f'path/to/my/folder/{well}(fld {field} wv TL-Brightfield - dsRed)_mask.tif'
    label_map_0610 = f'path/to/my/folder/{well}(fld {field} wv TL-Brightfield - dsRed)_mask.tif'

    img_0529 = plt.imread(label_map_0529)
    img_0610 = plt.imread(label_map_0610)
    
    correspondence = np.unique(img_0529.astype(np.complex64) + 1j * img_0610.astype(np.complex64))
    correspondence = correspondence[correspondence.real * correspondence.imag != 0]
    
    correspondence_list = [(int(np.round(c.real)), int(np.round(c.imag))) for c in correspondence]
    real_counts = Counter(pair[0] for pair in correspondence_list)
    imag_counts = Counter(pair[1] for pair in correspondence_list)

    field_correspondence[common_field] = [
        pair for pair in correspondence_list
        if real_counts[pair[0]] == 1 and imag_counts[pair[1]] == 1
    ]    

In [14]:
df_0529["is_shared"] = False
df_0610["is_shared"] = False

In [15]:
def get_corresponding_id(row, idx):
    field_name = row["field_name"]
    corresponding_pairs = np.array(field_correspondence.get(field_name, [])).astype(int)
    corresponding_pairs_dict = {row[idx%2]: row[(idx+1)%2] for row in corresponding_pairs}
    # 対応するペアがない時はNone
    row["corresponding_pair_id"] = corresponding_pairs_dict.get(row['bar_id'], None)
    row["is_shared"] = row["corresponding_pair_id"] is not None
    if idx == 0:
        df = df_0610
    else:
        df = df_0529
    if len(corresponding_df := df[(df["field_name"] == field_name) & (df["bar_id"] == row["corresponding_pair_id"])]) > 0:
        row["corresponding_pair_hash"] = str(corresponding_df["hash_simple"].values[0]).zfill(5)
    return row

In [16]:
df_0529 = df_0529.apply(lambda row:get_corresponding_id(row, 0), axis=1)
df_0610 = df_0610.apply(lambda row:get_corresponding_id(row, 1), axis=1)

In [ ]:
df_0529.to_csv("../output/0529_with_positions_correspondence.csv", index=False)
df_0610.to_csv("../output/0610_with_positions_correspondence.csv", index=False)

In [45]:
print("Date: 0529")
print("total: ", df_0529.shape[0]//5)
filtered_df_0529 = df_0529[(df_0529["is_shared"]) & (df_0529["corresponding_pair_hash"].apply(lambda x: isinstance(x, str)))]
print("shared: ", filtered_df_0529.shape[0]//5)

print("")
print("Date: 0610")
print("total: ", df_0610.shape[0]//5)
filtered_df_0610 = df_0610[(df_0610["is_shared"]) & (df_0610["corresponding_pair_hash"].apply(lambda x: isinstance(x, str)))]
print("shared: ", filtered_df_0610.shape[0]//5)

Date: 0529
total:  16174
shared:  10654

Date: 0610
total:  16850
shared:  10654


In [27]:
df_0529.loc[:,["layer_id"]+ [f"probabilities_{i}" for i in range(1, 10)]+[f"ellipse_id_layer_{i}" for i in range(1, 6)]]

,layer_id,probabilities_1,probabilities_2,probabilities_3,probabilities_4,probabilities_5,probabilities_6,probabilities_7,probabilities_8,probabilities_9,ellipse_id_layer_1,ellipse_id_layer_2,ellipse_id_layer_3,ellipse_id_layer_4,ellipse_id_layer_5
0,Layer 1,1.774605e-17,5.025913e-40,8.723668e-28,2.857516e-51,3.674432e-58,1.537256e-07,9.999998e-01,6.925124e-35,7.333587e-20,0.0,0.0,7.0,0.0,0.0
1,Layer 2,8.180713e-124,3.358238e-160,9.887292e-138,3.323673e-14,4.351092e-196,3.313046e-169,1.701676e-62,3.978819e-304,1.000000e+00,0.0,0.0,7.0,0.0,0.0
2,Layer 3,2.538248e-45,4.192690e-21,1.464937e-35,9.919278e-01,2.382016e-07,3.206703e-27,2.168582e-29,1.497866e-68,8.071923e-03,0.0,0.0,7.0,0.0,0.0
3,Layer 4,7.295288e-42,1.000000e+00,1.642271e-36,2.932771e-37,4.262903e-42,1.204873e-49,1.482144e-27,1.334313e-09,8.817745e-31,0.0,0.0,7.0,0.0,0.0
4,Layer 5,4.145119e-169,7.323355e-97,1.000000e+00,4.383579e-140,1.803986e-268,2.237877e-202,7.832913e-185,2.339975e-13,1.458947e-41,0.0,0.0,7.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80865,Layer 1,4.707684e-87,9.997776e-01,1.604510e-69,1.008534e-42,1.557642e-18,5.413227e-89,4.019910e-163,2.223976e-04,3.887669e-145,4.0,1.0,6.0,9.0,5.0
80866,Layer 2,1.000000e+00,3.679714e-43,2.793127e-90,2.623569e-31,8.282485e-89,3.226437e-50,2.481599e-63,2.649004e-97,3.855530e-71,4.0,1.0,6.0,9.0,5.0
80867,Layer 3,1.262955e-16,2.159576e-11,4.027917e-03,3.792441e-15,3.317653e-27,5.878210e-09,4.296894e-06,9.959678e-01,1.193354e-08,4.0,1.0,6.0,9.0,5.0
80868,Layer 4,9.419928e-45,9.655562e-21,3.068694e-65,6.651517e-38,1.955769e-15,4.131555e-33,1.000000e+00,6.238448e-19,5.199626e-32,4.0,1.0,6.0,9.0,5.0
